In [ ]:
import pandas as pd
import numpy as np
import keras
import seaborn as sns
np.random.seed(2)

In [ ]:
data = pd.read_csv('training_set.csv', index_col = 0)

##?Data exploration

In [ ]:
data.head()

In [ ]:
from scipy.stats import pearsonr
cor, _ = pearsonr(data['Time'], data['Class'])
print(cor)

##?Pre-processing

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
data['Mud'] = scaler.fit_transform(data[['Mud']])
#data = data.drop(['Amount'],axis=1)

In [ ]:
data.head()

In [ ]:
data = data.drop(['Time'],axis=1)
data.head()

In [ ]:
X = data.iloc[:, data.columns != 'Class']
y = data.iloc[:, data.columns == 'Class']

In [ ]:
y.head(10)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=0)
y_np = np.array(y)
X_np = np.array(X)

In [ ]:
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

In [ ]:
def confusion_matrix(y_pred, y_np, str):
    s = y_pred.shape
    pred = np.zeros((2, 2))
    lis = []
    for i in range(1, s[0]):
      pred[int(y_pred[i].round()), y_np[i]] = pred[int(y_pred[i].round()), y_np[i]]+1
    pred = np.transpose(pred)
    fig, ap = plt.subplots(figsize = (5,5))
    ax = sns.heatmap(pred, annot=True, cmap='Blues', ax = ap, fmt = '1')

    ax.set_title('Confusion Matrix ' + str +'\n\n');
    ax.set_xlabel('\nPredicted Values')
    ax.set_ylabel('Actual Values ');

    ## Ticket labels - List must be in alphabetical order
    ax.xaxis.set_ticklabels(['\n0', '\n1'])
    ax.yaxis.set_ticklabels(['0', '1'])
    ## Display the visualization of the Confusion Matrix.
    plt.show()

# Decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
decision_tree = DecisionTreeClassifier()
decision_tree.fit(X_train,y_train.ravel())

In [ ]:
y_pred = decision_tree.predict(X_test)
confusion_matrix(y_pred, y_test, 'for test dataset')

In [ ]:
print('Test score:', decision_tree.score(X_test,y_test))

In [ ]:
y_pred = decision_tree.predict(X)
confusion_matrix(y_pred, y_np, 'for whole dataset')

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
random_forest = RandomForestClassifier(n_estimators=100)
random_forest.fit(X_train,y_train.ravel())

In [ ]:
y_pred = random_forest.predict(X_test)
confusion_matrix(y_pred, y_test, 'for test dataset')

In [ ]:
print('Test score:', random_forest.score(X_test,y_test))

In [ ]:
y_pred = random_forest.predict(X)
confusion_matrix(y_pred, y_np, 'for whole dataset')

## XGBoost

In [ ]:
from xgboost import XGBClassifier
classifier = XGBClassifier()
classifier.fit(X_train, y_train)

In [ ]:
y_pred = classifier.predict(X_test)
confusion_matrix(y_pred, y_test, 'for test dataset')

In [ ]:
print('Test score:', classifier.score(X_test,y_test))

In [ ]:
y_pred = classifier.predict(X)
confusion_matrix(y_pred, y_np, 'for whole dataset')

## Deep neural network

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout

In [ ]:
model = Sequential([
    Dense(units=16, input_dim = 29,activation='relu'),
    Dense(units=24,activation='relu'),
    Dropout(0.5),
    Dense(20,activation='relu'),
    Dense(24,activation='relu'),
    Dense(1,activation='sigmoid'),
])

In [ ]:
model.summary()

##?Training

In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.fit(X_train,y_train,batch_size=15,epochs=5)

In [ ]:
score = model.evaluate(X_test, y_test)

In [ ]:
print(score)

In [ ]:
y_pred.round()

In [ ]:
y_pred = model.predict(X_test)
confusion_matrix(y_pred, y_test, 'for test dataset')

In [ ]:
y_pred = model.predict(X)
confusion_matrix(y_pred, y_np, 'for whole dataset')